In [1]:
'''Imports & Setup
Goal: Load the data engineering stack.

Note: You’ll need pandas and numpy. I've also included re for the heavy lifting of text cleaning.'''

import pandas as pd
import numpy as np
import re
import io
from typing import List, Dict

print("✅ Data Engineering stack loaded.")

✅ Data Engineering stack loaded.


In [8]:
'''Create "Dirty" Mock Dataset
Goal: Create a dataset that reflects common real-world issues: missing values, HTML tags, extra whitespace, and inconsistent casing.'''

# Simulating a raw dataset of customer queries for an LLM chatbot
raw_data = """id,timestamp,raw_text
1,2026-05-12 10:00:05,  "  How do I reset my PW??  "
2,2026-05-12 10:05:10, <div>Check status of order #12345</div>
3,2026-05-12 10:10:00, 
4,2026-05-12 10:15:20, "PLEASE HELP!!! My app is crashing... 🚀"
5,2026-05-12 10:20:00, "How to use the <b>API</b>?"
"""

# Load into DataFrame
df = pd.read_csv(io.StringIO(raw_data))
print("📊 Raw Data Loaded:")
display(df)

📊 Raw Data Loaded:


,id,timestamp,raw_text
0,1,2026-05-12 10:00:05,""" How do I reset my PW?? """
1,2,2026-05-12 10:05:10,<div>Check status of order #12345</div>
2,3,2026-05-12 10:10:00,
3,4,2026-05-12 10:15:20,"""PLEASE HELP!!! My app is crashing... 🚀"""
4,5,2026-05-12 10:20:00,"""How to use the <b>API</b>?"""


In [9]:
''' Preprocessing Module:
Goal: Define reusable functions for cleaning and normalization.
Pattern: We use a "Pipeline" approach where functions are modular and testable.'''

def clean_text(text: str) -> str:
    """Removes HTML tags, special characters, and normalizes whitespace."""
    if not isinstance(text, str) or text.strip() == "":
        return ""
    
    # 1. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # 2. Remove non-alphanumeric (keep basic punctuation)
    text = re.sub(r'[^a-zA-Z0-9\s?\.!#]', '', text)
    # 3. Normalize whitespace and casing
    text = " ".join(text.split()).lower()
    
    return text

def get_text_stats(text: str) -> Dict:
    """Feature engineering: extracts basic metadata from text."""
    return {
        "char_count": len(text),
        "word_count": len(text.split()),
        "has_id_ref": 1 if "#" in text else 0
    }

print("✅ Preprocessing functions defined.")


✅ Preprocessing functions defined.


In [ ]:

# 1. Cleaning logic 
df['cleaned_text'] = df['raw_text'].apply(clean_text)

# 2. Statistics logic 

stats_columns = df['cleaned_text'].apply(lambda x: pd.Series(get_text_stats(x)))
df = pd.concat([df, stats_columns], axis=1)

print("✅ Columns created:", df.columns.tolist())

✅ Columns created: ['id', 'timestamp', 'raw_text', 'cleaned_text', 'char_count', 'word_count', 'has_id_ref']


In [11]:
'''Applying Transformations
Goal: Execute the cleaning and engineer new features (metadata) useful for model filtering'''

# 1. Clean the text column
df['cleaned_text'] = df['raw_text'].apply(clean_text)

# 2. Filter out empty records (Common in LLM fine-tuning prep)
df = df[df['cleaned_text'] != ""].copy()

# 3. Feature Engineering: Extracting metadata
stats_df = df['cleaned_text'].apply(pd.Series(get_text_stats))
df = pd.concat([df, stats_df], axis=1)

print("✅ Transformations applied.")

✅ Transformations applied.


In [12]:
'''The "Before-Versus-After" Summary
Goal: Provide the high-level summary requested in the sprint "Output" section. This is a great screenshot for your proof pack.'''

print("--- 📋 Dataset Summary ---")
print(f"Total Rows Processed: {len(df)}")
print(f"Empty Rows Dropped: {len(pd.read_csv(io.StringIO(raw_data))) - len(df)}")

print("\n--- 🔍 Before vs After Sample ---")
comparison = df[['raw_text', 'cleaned_text', 'word_count']].head()
display(comparison)

# Export for Deliverable
df.to_csv("processed_llm_data.csv", index=False)
print("\n✅ Cleaned dataset exported to 'processed_llm_data.csv'")

--- 📋 Dataset Summary ---
Total Rows Processed: 8
Empty Rows Dropped: -3

--- 🔍 Before vs After Sample ---


,raw_text,cleaned_text,cleaned_text,word_count
0,""" How do I reset my PW?? """,how do i reset my pw??,NaN,6.0
1,<div>Check status of order #12345</div>,check status of order #12345,NaN,5.0
3,"""PLEASE HELP!!! My app is crashing... 🚀""",please help!!! my app is crashing...,NaN,6.0
4,"""How to use the <b>API</b>?""",how to use the api?,NaN,5.0
"(0, 0)",NaN,NaN,"{'char_count': 22, 'word_count': 6, 'has_id_re...",NaN



✅ Cleaned dataset exported to 'processed_llm_data.csv'


In [13]:
'''Unit tests for the cleaning function'''

def run_tests():
    test_html = "Hello <b>World</b>"
    test_empty = "   "
    
    assert clean_text(test_html) == "hello world"
    assert clean_text(test_empty) == ""
    
    print("✅ All Unit Tests Passed: logic is sound for production.")

run_tests()

✅ All Unit Tests Passed: logic is sound for production.
